In [1]:
import pandas as pd

df = pd.read_csv('../data/sample_10k_maps_with_url_status.csv')

In [3]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detect_language_safe(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'unknown'
    try:
        lang = detect(str(text))
        if lang in ['zh-tw', 'zh-hk']:
            return 'zh-cn'
        return lang
    except:
        return 'unknown'

In [4]:
print("Check text languages...")
df['detected_lang'] = df['text'].apply(detect_language_safe)

print("\nLanguage distribution for all URLs:")
print(df['detected_lang'].value_counts())

Check text languages...

Language distribution for all URLs:
detected_lang
en         4188
fr          532
de          514
ko          321
ru          317
es          306
tl          295
it          294
nl          247
unknown     240
sw          231
ja          225
zh-cn       206
pt          165
id          160
ca          140
ro          137
pl          118
no          114
af          106
vi          104
sl          101
da           90
bg           80
sv           75
so           72
tr           62
ar           52
et           48
hu           46
cy           43
cs           42
hr           41
fi           40
mk           37
lt           31
sk           29
el           26
fa           22
sq           21
uk           21
he           16
lv           14
th           12
bn           10
ta            4
ur            3
hi            2
ne            1
mr            1
Name: count, dtype: int64


In [14]:
all_languages = df['detected_lang'].unique().tolist()
ratio_per_language_in_whole_df = {
    lang: len(df[df['detected_lang'] == lang]) / len(df) * 100 for lang in all_languages
}

ratio_df = pd.DataFrame.from_dict(ratio_per_language_in_whole_df, orient='index', columns=['Ratio (%)'])

language_code_to_language_name = {
    'en': 'English',
    'fr': 'French',
    'de': 'German',
    'ko': 'Korean',
    'ru': 'Russian',
    'es': 'Spanish',
    'tl': 'Filipino',
    'it': 'Italian',
    'nl': 'Dutch',
    'unknown': 'Unknown',
    'sw': 'Swahili',
    'ja': 'Japanese',
    'zh-cn': 'Simplified Chinese',
    'pt': 'Portuguese',
    'id': 'Indonesian',
    'ca': 'Catalan',
    'ro': 'Romanian',
    'pl': 'Polish',
    'no': 'Norwegian',
    'af': 'Afrikaans',
    'vi': 'Vietnamese',
    'sl': 'Slovenian',
    'da': 'Danish',
    'bg': 'Bulgarian',
    'sv': 'Swedish',
    'so': 'Somali',
    'tr': 'Turkish',
    'ar': 'Arabic',
    'et': 'Estonian',
    'hu': 'Hungarian',
    'cy': 'Welsh',
    'cs': 'Czech',
    'hr': 'Croatian',
    'fi': 'Finnish',
    'mk': 'Macedonian',
    'lt': 'Lithuanian',
    'sk': 'Slovak',
    'el': 'Greek',
    'fa': 'Persian',
    'sq': 'Albanian',
    'uk': 'Ukrainian',
    'he': 'Hebrew',
    'lv': 'Latvian',
    'th': 'Thai',
    'bn': 'Bengali',
    'ta': 'Tamil',
    'ur': 'Urdu',
    'hi': 'Hindi',
    'ne': 'Nepali',
    'mr': 'Marathi',
}

ratio_df['Language'] = ratio_df.index.map(language_code_to_language_name)
ratio_df = ratio_df.sort_values(by='Ratio (%)', ascending=False)
print("\nLanguage distribution with names:")
print(ratio_df)


Language distribution with names:
         Ratio (%)            Language
en       41.871626             English
fr        5.318936              French
de        5.138972              German
ko        3.209358              Korean
ru        3.169366             Russian
es        3.059388             Spanish
tl        2.949410            Filipino
it        2.939412             Italian
nl        2.469506               Dutch
unknown   2.399520             Unknown
sw        2.309538             Swahili
ja        2.249550            Japanese
zh-cn     2.059588  Simplified Chinese
pt        1.649670          Portuguese
id        1.599680          Indonesian
ca        1.399720             Catalan
ro        1.369726            Romanian
pl        1.179764              Polish
no        1.139772           Norwegian
af        1.059788           Afrikaans
vi        1.039792          Vietnamese
sl        1.009798           Slovenian
da        0.899820              Danish
bg        0.799840           

In [37]:
def check_threshold(threshold=1.0):
    covered_ratio = ratio_df[ratio_df['Ratio (%)'] >= threshold]['Ratio (%)'].sum()
    print(f"Percentage of languages covering at least {threshold}: {covered_ratio:.2f}%")

    num_languages_above_threshold = len(ratio_df[ratio_df['Ratio (%)'] >= threshold])
    num_languages_below_threshold = len(ratio_df[ratio_df['Ratio (%)'] < threshold])
    print(f"Number of languages with at least {threshold}%: {num_languages_above_threshold}")
    print(f"Number of languages with less than {threshold}%: {num_languages_below_threshold}")

In [38]:
check_threshold(1)

Percentage of languages covering at least 1: 90.59%
Number of languages with at least 1%: 22
Number of languages with less than 1%: 28


In [39]:
check_threshold(2)

Percentage of languages covering at least 2: 79.14%
Number of languages with at least 2%: 13
Number of languages with less than 2%: 37


In [40]:
check_threshold(1.5)

Percentage of languages covering at least 1.5: 82.39%
Number of languages with at least 1.5%: 15
Number of languages with less than 1.5%: 35


Wybieramy threshold 1%, co oznacza, że języki z mniej niż 1% udziału zostaną zsumowane do kategorii "Inne" i nie utworzymy słowników dla tych języków.

In [46]:
selected_languages = ratio_df[ratio_df['Ratio (%)'] >= 1].index.tolist()
print(f"Selected languages for dictionaries: {selected_languages}")

selected_languages_without_unknown = [lang for lang in selected_languages if lang != 'unknown']

covered_ratio = ratio_df[ratio_df.index.isin(selected_languages_without_unknown)]['Ratio (%)'].sum()
print(f"Percentage of maps covered by selected languages without unknown: {covered_ratio:.2f}%")

Selected languages for dictionaries: ['en', 'fr', 'de', 'ko', 'ru', 'es', 'tl', 'it', 'nl', 'unknown', 'sw', 'ja', 'zh-cn', 'pt', 'id', 'ca', 'ro', 'pl', 'no', 'af', 'vi', 'sl']
Percentage of maps covered by selected languages without unknown: 88.19%


Przygotujemy zatem słowniki słów pozytywnych i negatywnych w językach: ['en', 'fr', 'de', 'ko', 'ru', 'es', 'tl', 'it', 'nl', 'sw', 'ja', 'zh-cn', 'pt', 'id', 'ca', 'ro', 'pl', 'no', 'af', 'vi', 'sl'].

Na wszystkich mapach stosujemy wszystkie słowniki - aby wychwycić mieszanie języków + detector nie zawsze dobrze sobie radzi przy krótkich tekstach.

In [62]:
# Positive keywords
positive_keywords = {
    # === (5 pkt) ===
    'choropleth': 5,
    'cartogram': 5,
    'proportional symbol': 5,
    'graduated symbol': 5,
    'graduated circles': 5,
    'statistical map': 5,

    # === (4 pkt) ===
    'per capita': 4,
    'share of': 4,
    'proportion of': 4,
    'distribution of': 4,
    'breakdown by': 4,
    'by country': 4,
    'by age': 4,
    'by region': 4,
    'by province': 4,
    'by district': 4,
    'by municipality': 4,
    'by county': 4,
    'by state': 4,
    'per 1000': 4,
    'per 100k': 4,
    'life expectancy': 4,
    'growth rate': 4,

    # === (3 pkt) ===
    'demographic': 3,
    'demographical': 3,
    'statistical': 3,
    'quantitative': 3,
    'census': 3,
    'prevalence': 3,
    'disparity': 3,
    'inequality': 3,
    'standardised': 3,
    'classification': 3,
    'GDP': 3,

    # === (2 pkt) ===
    'population': 2,
    'density': 2,
    'mortality': 2,
    'fertility': 2,
    'migration': 2,
    'unemployment': 2,
    'poverty': 2,
    'wealth': 2,
    'literacy': 2,
    'urbanisation': 2,
    'employment': 2,
    'labor force': 2,
    'labor': 2,
    'productivity': 2,
    'median': 2,
    'mean': 2,
    'quintile': 2,
    'quartile': 2,
    'percentile': 2,
    'coefficient': 2,
    'correlation': 2,
    'deviation': 2,
    'ratio': 2,
    'indicator': 2,
    'income': 2,
    'salary': 2,
    'wage': 2,
    'wages': 2,
    'earnings': 2,
    'expenditure': 2,
    'revenue': 2,
    'budget': 2,
    'investment': 2,
    'R&D': 2,
    'comparison': 2,
    'thematic': 2,
    'survey': 2,
    'bureau': 2,
    'average': 2,

    # === (1 pkt) ===
    'county': 1,
    'counties': 1,
    'country': 1,
    'region': 1,
    'municipality': 1,
    'state': 1,
    'states': 1,
    'province': 1,
    'provinces': 1,
    'district': 1,
    'districts': 1,
    'prefecture': 1,
    'canton': 1,
    'oblast': 1,
    'commune': 1,
    'constituency': 1,
    'ward': 1,
    'borough': 1,
    'subdivision': 1,
    'voivodship': 1,
    'voivodships': 1,
    'NUTS': 1,
    'administrative': 1,
    'people': 1,
    'men': 1,
    'women': 1,
    'gender': 1,
    'sex': 1,
    'age': 1,
    'aged': 1,
    'birth': 1,
    'deaths': 1,
    'inhabitant': 1,
    'household': 1,
    'immigrant': 1,
    'emigration': 1,
    'education': 1,
    'health': 1,
    'disease': 1,
    'obesity': 1,
    'smoking': 1,
    'voting': 1,
    'election': 1,
    'ethnic': 1,
    'ethnicity': 1,
    'language': 1,
    'religion': 1,
    'tax': 1,
    'taxes': 1,
    'sector': 1,
    'industry': 1,
    'agriculture': 1,
    'services': 1,
    'manufacturing': 1,
    'rate': 1,
    'rates': 1,
    'percentage': 1,
    'percent': 1,
    'index': 1,
    'variation': 1,
    'decline': 1,
    'change': 1,
    'trend': 1,
    'breakdown': 1,
    'stats': 1,
    'global': 1,
    'annual': 1,
    'economic': 1,
    'unit': 1,
    'units': 1,
    'expectancy': 1,
    'outcome': 1,

    # === (0.5 pkt) ===
    'chart': 0.5,
    'pie': 0.5,
    'legend': 0.5,
    'area': 0.5,
    'size': 0.5,
    'data': 0.5,
    'number of': 0.5,
    'sum': 0.5,
    'net': 0.5,
}

print(f"Total positive keywords: {len(positive_keywords)}")

csv_pos_content = "key,score,en\n"
for keyword, score in positive_keywords.items():
    key = f"{keyword.replace(' ', '_').lower()}"
    value = keyword
    csv_pos_content += f"{key},{score},{value}\n"

with open("./keywords/positive_keywords.csv", "w", encoding="utf-8") as f:
    f.write(csv_pos_content)

Total positive keywords: 154


In [61]:
# Negative keywords
negative_keywords = {
    # === (-5 pkt) ===
    'floor plan': -5,
    'floorplan': -5,
    'ground floor': -5,
    'floor plans': -5,
    'blueprint': -5,
    'elevation': -5,
    'cross-section': -5,
    'architectural': -5,
    'architecture': -5,
    'interior': -5,
    'stock photo': -5,
    'royalty free': -5,
    'photography': -5,
    'aerial view': -5,
    'satellite image': -5,
    'satellite imagery': -5,
    'old map': -5,
    'vintage map': -5,
    'antique map': -5,
    'historical map': -5,
    'vintage': -5,

    # === (-4 kt) ===
    'road map': -4,
    'street map': -4,
    'route map': -4,
    'transit map': -4,
    'subway map': -4,
    'metro map': -4,
    'bus route': -4,
    'railway map': -4,
    'highway map': -4,
    'navigation': -4,
    'travel map': -4,
    'things to do': -4,
    'attractions': -4,
    'sightseeing': -4,
    'landmarks': -4,
    'visitor guide': -4,
    'history': -4,
    'historical': -4,
    'clipart': -4,
    'vector graphic': -4,
    'illustration': -4,
    'cartoon': -4,
    'drawing': -4,
    'sketch': -4,
    'rendering': -4,
    '3d model': -4,
    '3d rendering': -4,
    '3d': -4,
    'surface': -4,
    'clicking': -4,
    'illustratie': -4,
    'topographic': -4,
    'topographical': -4,
    'relief map': -4,
    'terrain map': -4,
    'physical map': -4,
    'contour map': -4,
    'geological': -4,

    # === (-3 pkt) ===
    'photo': -3,
    'photos': -3,
    'photograph': -3,
    'image': -3,
    'images': -3,
    'screenshot': -3,
    'foto': -3,
    'fotos': -3,
    'thumbnail': -3,
    'labeled continents': -3,
    'icon': -3,
    'icons': -3,
    'logo': -3,
    'badge': -3,
    'emblem': -3,
    'coat of arms': -3,
    'flag': -3,
    'flags': -3,
    'vectorpictogram': -3,
    'wallpaper': -3,
    'background': -3,
    'texture': -3,
    'pattern': -3,
    'poster': -3,
    'banner': -3,
    'weather map': -3,
    'cloud cover': -3,
    'temperature map': -3,
    'climatic': -3,
    'climate map': -3,
    'video': -3,
    'animation': -3,
    'animated': -3,
    'footage': -3,

    # === (-2 pkt) ===
    'building': -2,
    'buildings': -2,
    'apartment': -2,
    'apartments': -2,
    'residential': -2,
    'parking': -2,
    'garage': -2,
    'satellite': -2,
    'road': -2,
    'roads': -2,
    'route': -2,
    'street': -2,
    'streets': -2,
    'highway': -2,
    'motorway': -2,
    'metro': -2,
    'subway': -2,
    'railway': -2,
    'railroad': -2,
    'hotel': -2,
    'hotels': -2,
    'restaurant': -2,
    'beach': -2,
    'mountain': -2,
    'trail': -2,
    'trails': -2,
    'hiking': -2,
    'landmark': -2,
    'layout': -2,
    'postcard': -2,
    'stamp': -2,
    'stamps': -2,
    'printable': -2,
    'print': -2,
    'encyclopedia britannica': -2,
    'wikipedia': -2,

    # === (-1 pkt) ===
    'coverage map': -1,
    'plan': -1,
    'plans': -1,
    'planos': -1,
    'pinterest': -1,
    'guide': -1,
    'travel': -1,
    'visit': -1,
    'vacation': -1,
    'tourist': -1,
    'tourism': -1,

    # === (-0.5 pkt) ===
    'km': -0.5,
    'miles': -0.5,
    'meters': -0.5,
    'precipitation': -0.5,
    'city': -0.5,
    'cities': -0.5,
    'town': -0.5,
    'urban': -0.5,
    'village': -0.5,
    'villages': -0.5,
    'suburb': -0.5,
    'suburbs': -0.5,
}

print(f"Total negative keywords: {len(negative_keywords)}")

csv_neg_content = "key,score,en\n"
for keyword, score in negative_keywords.items():
    key = f"{keyword.replace(' ', '_').lower()}"
    value = keyword
    csv_neg_content += f"{key},{score},{value}\n"

with open("./keywords/negative_keywords.csv", "w", encoding="utf-8") as f:
    f.write(csv_neg_content)

Total negative keywords: 152


In [71]:
positive_keywords_df = pd.read_csv("./keywords/positive_keywords.csv", index_col=0)
positive_keywords_df.head()

,score,en
key,,
choropleth,5.0,choropleth
cartogram,5.0,cartogram
proportional_symbol,5.0,proportional symbol
graduated_symbol,5.0,graduated symbol
graduated_circles,5.0,graduated circles


In [72]:
pos_fr = pd.read_csv("positive_keywords_fr.csv", index_col=0)
pos_es = pd.read_csv("positive_keywords.csv", index_col=0)

positive_keywords_df = positive_keywords_df.merge(pos_fr[['fr']], left_index=True, right_index=True, how='left')
positive_keywords_df = positive_keywords_df.merge(pos_es[['es']], left_index=True, right_index=True, how='left')
positive_keywords_df

,score,en,fr,es
key,,,,
choropleth,5.0,choropleth,choroplethe,coropleta
cartogram,5.0,cartogram,cartogramme,cartograma
proportional_symbol,5.0,proportional symbol,symbole proportionnel,símbolo proporcional
graduated_symbol,5.0,graduated symbol,symbole gradué,símbolo graduado
graduated_circles,5.0,graduated circles,cercles gradués,círculos graduados
...,...,...,...,...
size,0.5,size,taille,tamaño
data,0.5,data,données,datos
number_of,0.5,number of,nombre de,número de


In [73]:
positive_keywords_df.to_csv("./keywords/positive_keywords.csv")

In [92]:
# TRANSLATION FUNCTION
from googletrans import Translator

def translate_to_lang(data, dest_lang):
    translator = Translator()

    def translate_column(text):
        translated = translator.translate(text, dest=dest_lang)
        return translated.text.lower()

    data[dest_lang] = data['en'].apply(translate_column)

In [104]:
# FLAT KEYWORDS JSON
import json

def make_flat_json(data, output_file):
    cols_to_iterate = [col for col in data.columns if col not in ['key', 'score']]

    flat_json = {}
    for _, row in data.iterrows():
        scr = row['score']
        for col in cols_to_iterate:
            word = row[col]
            flat_json[word] = scr

    with open(output_file, "w", encoding="utf-8") as flat:
        json.dump(flat_json, flat, ensure_ascii=False, indent=2)

In [101]:
# POSITIVE KEYWORDS TRANSLATION

# languages = ['fr', 'de', 'ko', 'ru', 'es', 'tl', 'it', 'nl', 'sw', 'ja', 'zh-cn', 'pt', 'id', 'ca', 'ro', 'pl', 'no', 'af', 'vi', 'sl']
languages = ['ca', 'ro', 'pl', 'sl', 'zh-cn', 'pt', 'id', 'no', 'af', 'vi']

for lang in languages:
    start_time = pd.Timestamp.now()
    print(f"Translating to {lang}...")
    translate_to_lang(positive_keywords_df, lang)

    end_time = pd.Timestamp.now()
    print(f"Translation to {lang} completed in {end_time - start_time}.")

    print(f"Number of translated keywords so far: {positive_keywords_df[lang].notna().sum()}")
    positive_keywords_df.to_csv("./keywords/positive_keywords_iterator.csv")

Translating to ca...
Translation to ca completed in 0 days 00:01:40.132227.
Number of translated keywords so far: 154
Translating to ro...
Translation to ro completed in 0 days 00:01:19.149617.
Number of translated keywords so far: 154
Translating to pl...
Translation to pl completed in 0 days 00:00:46.251930.
Number of translated keywords so far: 154
Translating to sl...
Translation to sl completed in 0 days 00:00:57.000831.
Number of translated keywords so far: 154
Translating to zh-cn...
Translation to zh-cn completed in 0 days 00:01:19.907109.
Number of translated keywords so far: 154
Translating to pt...
Translation to pt completed in 0 days 00:01:44.313144.
Number of translated keywords so far: 154
Translating to id...
Translation to id completed in 0 days 00:01:45.724226.
Number of translated keywords so far: 154
Translating to no...
Translation to no completed in 0 days 00:01:34.384518.
Number of translated keywords so far: 154
Translating to af...
Translation to af completed i

In [103]:
positive_keywords_df.to_csv("./keywords/positive_keywords_final.csv")

In [212]:
make_flat_json(positive_keywords_df, "./keywords/positive_keywords.json")

In [216]:
negative_keywords_df = pd.read_csv("./keywords/negative_keywords.csv", index_col=0)

negative_keywords_df.head()

,{
"""floor plan"": -5",NaN
"""floorplan"": -5",NaN
"""ground floor"": -5",NaN
"""floor plans"": -5",NaN
"""blueprint"": -5",NaN


In [109]:
# NEGATIVE KEYWORDS TRANSLATION

languages = ['fr', 'es', 'de', 'ko', 'ru', 'tl', 'it', 'nl', 'sw', 'ja', 'ca', 'ro', 'pl', 'sl', 'zh-cn', 'pt', 'id', 'no', 'af', 'vi']

start_time_all = pd.Timestamp.now()
for lang in languages:
    start_time = pd.Timestamp.now()
    print(f"Translating to {lang}...")
    translate_to_lang(negative_keywords_df , lang)

    end_time = pd.Timestamp.now()
    print(f"Translation to {lang} completed in {end_time - start_time}.")

    print(f"Number of translated keywords so far: {negative_keywords_df[lang].notna().sum()}")
    negative_keywords_df.to_csv("./keywords/negative_keywords_iterator.csv")

    print("---------------")

end_time_all = pd.Timestamp.now()

print(f"All translations completed in {end_time_all - start_time_all}.")

negative_keywords_df.to_csv("./keywords/negative_keywords_final.csv")

Translating to fr...
Translation to fr completed in 0 days 00:00:53.417945.
Number of translated keywords so far: 152
---------------
Translating to es...
Translation to es completed in 0 days 00:00:43.129487.
Number of translated keywords so far: 152
---------------
Translating to de...
Translation to de completed in 0 days 00:00:34.325958.
Number of translated keywords so far: 152
---------------
Translating to ko...
Translation to ko completed in 0 days 00:01:41.017832.
Number of translated keywords so far: 152
---------------
Translating to ru...
Translation to ru completed in 0 days 00:00:29.590443.
Number of translated keywords so far: 152
---------------
Translating to tl...
Translation to tl completed in 0 days 00:01:05.920671.
Number of translated keywords so far: 152
---------------
Translating to it...
Translation to it completed in 0 days 00:01:19.329811.
Number of translated keywords so far: 152
---------------
Translating to nl...
Translation to nl completed in 0 days 00:

In [219]:
make_flat_json(negative_keywords_df, "./keywords/negative_keywords.json")

In [218]:
negative_keywords_df = pd.read_csv("./keywords/negative_keywords_final.csv", index_col=0)
negative_keywords_df

,score,en,fr,es,de,ko,ru,tl,it,nl,...,ca,ro,pl,sl,zh-cn,pt,id,no,af,vi
key,,,,,,,,,,,,,,,,,,,,,
floor_plan,-5.0,floor plan,plan d'étage,plano de planta,grundriss,평면도,план этажа,plano sa sahig,planimetria,plattegrond,...,planta,plan de etaj,plan piętra,tloris,平面图,planta baixa,denah lantai,planløsning,vloerplan,sơ đồ mặt bằng
floorplan,-5.0,floorplan,plan d'étage,plano de planta,grundriss,평면도,план этажа,floorplan,planimetria,plattegrond,...,plànol,plan de etaj,plan piętra,tloris,平面图,planta baixa,denah lantai,planløsning,vloerplan,sơ đồ mặt bằng
ground_floor,-5.0,ground floor,rez-de-chaussée,planta baja,erdgeschoss,1층,первый этаж,ground floor,pianterreno,begane grond,...,planta baixa,parter,parter,pritličje,一楼,térreo,lantai dasar,første etasje,grondvloer,tầng trệt
floor_plans,-5.0,floor plans,plans d'étage,planos de planta,grundrisse,평면도,планы этажей,mga plano sa sahig,planimetrie,plattegronden,...,plànols de planta,planuri de etaj,plany pięter,tlorisi,平面图,plantas baixas,denah lantai,plantegninger,vloerplanne,sơ đồ mặt bằng
blueprint,-5.0,blueprint,plan,cianotipo,entwurf,청사진,план,plano,progetto,blauwdruk,...,plànol,planul,plan,načrt,蓝图,planta,cetak biru,blåkopi,bloudruk,bản thiết kế
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
urban,-0.5,urban,urbain,urbano,urban,도시의,городской,urban,urbano,stedelijk,...,urbà,urban,miejski,mestni,城市的,urbano,perkotaan,urbane,stedelik,đô thị
village,-0.5,village,village,aldea,dorf,마을,деревня,nayon,villaggio,dorp,...,poble,sat,wieś,vas,村庄,aldeia,desa,landsby,dorpie,làng bản
villages,-0.5,villages,villages,pueblos,dörfer,마을,деревни,mga nayon,villaggi,dorpen,...,pobles,sate,wioski,vasi,村庄,aldeias,desa,landsbyer,dorpe,làng
